# 实验 S：深度薄层 × 类型 × 来源方向联合抽样

同步代码后先重启原 DeepSpatial Python 内核，再运行全部单元格，避免复用已加载的旧代码。默认顺序处理留出 1、2、3。

本实验加载 `outputs/v1_holdouts_1_2_3/holdout_i/model/model.ckpt` 中已有的 EMA 权重，不训练、不重新拟合 UOT 或生态位统计量。保留原生成总数、ODE 和表达稀疏化，仅改变来源与目标深度的抽样调度。

固定薄层中心为 0、10、20… µm，边界为中心±5 µm。小于一个细胞的期望允许生成 0，不强制补齐类型。

每轮先做 100 个随机种子的纯抽样检查，再执行一次完整体积推理。修改 GPU 后重启内核。


In [4]:
import os
import sys
from pathlib import Path

PROJECT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent / "HM_DeepSpital", parent)
     if (candidate / "examples/run_s.py").is_file()
     and (candidate / "deepspatial/core.py").is_file()),
    None,
)
if PROJECT is None:
    raise FileNotFoundError("请在 DeepSpatial 项目目录内打开此 Notebook。")
PROJECT = PROJECT.resolve()
sys.path.insert(0, str(PROJECT))

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # 必须在导入 torch 前设置
SOURCE_ROOT = PROJECT / "outputs/v1_holdouts_1_2_3"
OUTPUT_ROOT = PROJECT / "outputs/s_holdouts_1_2_3"
HOLDOUT_SLICE_IDS = [1, 2, 3]
SAMPLING_SEED = 42
DEPTH_BIN_WIDTH_UM = 10.0
DEPTH_GRID_ORIGIN_UM = 0.0
AUDIT_SEEDS = 100
AUDIT_ONLY = False  # True：仅检查抽样，不运行 ODE；该检查也需要全新输出目录
DEVICE = "auto"


In [5]:
from examples.run_s import run_experiment
import deepspatial as ds

if not Path(ds.__file__).resolve().is_relative_to(PROJECT):
    raise RuntimeError("内核已加载其他 deepspatial，请重启后运行全部单元格。")
print("代码：", ds.__file__)
print("已有模型：", SOURCE_ROOT)
print("实验 S 输出：", OUTPUT_ROOT)


代码： /private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/deepspatial/__init__.py
已有模型： /private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/v1_holdouts_1_2_3
实验 S 输出： /private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/s_holdouts_1_2_3


In [6]:
completed_runs = run_experiment(
    source_root=SOURCE_ROOT,
    output_root=OUTPUT_ROOT,
    holdouts=HOLDOUT_SLICE_IDS,
    sampling_seed=SAMPLING_SEED,
    bin_width_um=DEPTH_BIN_WIDTH_UM,
    grid_origin_um=DEPTH_GRID_ORIGIN_UM,
    device=DEVICE,
    audit_seeds=AUDIT_SEEDS,
    audit_only=AUDIT_ONLY,
)


Experiment S: holdout 1; checkpoint /private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/v1_holdouts_1_2_3/holdout_1/model/model.ckpt
Holdout 1: auditing 100 sampling seeds without ODE


Seed set to 42
DeepSpatial: 3D Reconstruct: 100%|██████████| 3/3 [03:56<00:00, 78.87s/gap, range=150.0-200.0um]


Experiment S completed: {'heldout_id': 1, 'generated_cells': 114757, 'class_switches': 0, 'output': '/private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/s_holdouts_1_2_3/holdout_1'}
Experiment S: holdout 2; checkpoint /private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/v1_holdouts_1_2_3/holdout_2/model/model.ckpt
Holdout 2: auditing 100 sampling seeds without ODE


Seed set to 42
DeepSpatial: 3D Reconstruct: 100%|██████████| 3/3 [04:00<00:00, 80.27s/gap, range=150.0-200.0um]


Experiment S completed: {'heldout_id': 2, 'generated_cells': 112777, 'class_switches': 0, 'output': '/private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/s_holdouts_1_2_3/holdout_2'}
Experiment S: holdout 3; checkpoint /private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/v1_holdouts_1_2_3/holdout_3/model/model.ckpt
Holdout 3: auditing 100 sampling seeds without ODE


Seed set to 42
DeepSpatial: 3D Reconstruct: 100%|██████████| 3/3 [03:14<00:00, 64.97s/gap, range=100.0-200.0um]


Experiment S completed: {'heldout_id': 3, 'generated_cells': 113664, 'class_switches': 0, 'output': '/private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/s_holdouts_1_2_3/holdout_3'}


## 输出说明

每轮分别保存至 `OUTPUT_ROOT/holdout_i/`，已有目录会报错，不覆盖。

- `generated.h5ad`：完整体积。obs 保存目标 Z、薄层编号、来源 parent/方向/类型，可复算生成调度；最终类别不被预算强制覆盖。
- `sampling_budget.csv`：每区间、薄层、方向、来源类型的期望与实际整数预算。
- `sampling_counts.csv`：实际来源类别和生成类别计数，可检查类型切换。
- `sampling_audit.csv`：100 个纯抽样种子的计数均值、方差及零计数频率；不是空间恢复准确率。
- `run_config.json`：沿用的原运行参数、仅推理标记、抽样参数和原 checkpoint SHA-256。

`AUDIT_ONLY=True` 时只生成配置和抽样审计文件。结果仍按原协议使用隐藏层 Z±5 µm 评价，不使用隐藏真值分配预算。


In [7]:
for run in completed_runs:
    print(run)


{'heldout_id': 1, 'generated_cells': 114757, 'class_switches': 0, 'output': '/private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/s_holdouts_1_2_3/holdout_1'}
{'heldout_id': 2, 'generated_cells': 112777, 'class_switches': 0, 'output': '/private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/s_holdouts_1_2_3/holdout_2'}
{'heldout_id': 3, 'generated_cells': 113664, 'class_switches': 0, 'output': '/private_sdb/huangyingjie/Code/DeepSpatial/HM_DeepSpital/outputs/s_holdouts_1_2_3/holdout_3'}
